In [ ]:
# !pip install https://github.com/kyamagu/faiss-wheels/releases/download/v1.7.3/faiss_gpu-1.7.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

### simple faiss gpu

In [1]:
import faiss
import numpy as np
import time

# 1. Setup Parameters
d = 1536  # Vector dimension
nb = 151936  # Database size
nq = 100  # Number of query vectors

# 2. Generate Random Data
print("Generating random data...")
np.random.seed(42)
# Database vectors
xb = np.random.random((nb, d)).astype("float32")
# Query vectors
xq = np.random.random((nq, d)).astype("float32")

# 3. Build the Index on CPU and Move to GPU
print("Building the Faiss index...")

# Step 3a: Create a standard CPU index
cpu_index = faiss.IndexFlatL2(d)

# Step 3b: Create a GPU resource object
# This manages GPU memory and CUDA streams
res = faiss.StandardGpuResources()

# Step 3c: Use the resource object to convert the CPU index to a GPU index
gpu_id = 0  # Use the first GPU
gpu_index = faiss.index_cpu_to_gpu(res, gpu_id, cpu_index)

# 4. Add Vectors to the GPU Index
print(f"Adding {nb} vectors to the GPU index...")
start_time = time.time()
gpu_index.add(xb)
end_time = time.time()

print(f"Vectors added in {end_time - start_time:.2f} seconds.")
print(f"Total vectors in index: {gpu_index.ntotal}")

# 5. Perform the Search
k = 5  # Number of nearest neighbors to find for each query

print(f"\nSearching for the {k} nearest neighbors of {nq} queries...")
start_time = time.time()
distances, indices = gpu_index.search(xq, k)
end_time = time.time()

print(f"GPU search completed in {end_time - start_time:.4f} seconds.")

# 6. Display Results
print("\n--- Search Results ---")
print("Shape of returned indices:", indices.shape)
print("Shape of returned distances:", distances.shape)

for i in range(5):  # Print results for the first 5 queries
    print(f"\nQuery {i}:")
    print(f"  - Nearest neighbor indices: {indices[i]}")
    print(f"  - L2 distances: {distances[i]}")

Generating random data...
Building the Faiss index...
Adding 151936 vectors to the GPU index...
Vectors added in 0.06 seconds.
Total vectors in index: 151936

Searching for the 5 nearest neighbors of 100 queries...
GPU search completed in 0.0027 seconds.

--- Search Results ---
Shape of returned indices: (100, 5)
Shape of returned distances: (100, 5)

Query 0:
  - Nearest neighbor indices: [135698 116457 125758   1979  86973]
  - L2 distances: [222.04825 224.97195 225.40994 225.80557 227.41   ]

Query 1:
  - Nearest neighbor indices: [ 31338  64121 130275  84519 109251]
  - L2 distances: [223.87848 224.63702 225.35855 225.62106 225.91754]

Query 2:
  - Nearest neighbor indices: [ 98052 131774 139670  31560  88476]
  - L2 distances: [226.57025 227.45123 228.79678 229.67374 230.02173]

Query 3:
  - Nearest neighbor indices: [ 18834   5675  57672 145456   9385]
  - L2 distances: [230.09717 230.14667 230.73212 230.7691  230.83765]

Query 4:
  - Nearest neighbor indices: [ 26751  65606 1340

In [1]:
import faiss
import numpy as np
import time

# 1. Setup Parameters
d = 1536  # Vector dimension
nb = 151936  # Database size
nq = 100  # Number of query vectors

# 2. Generate Random Data
print("Generating random data...")
np.random.seed(42)
# Database vectors
xb = np.random.random((nb, d)).astype("float32")
# Query vectors
xq = np.random.random((nq, d)).astype("float32")
# Training vectors (a subset of the database is usually sufficient)
nt = 30000
xt = np.random.random((nt, d)).astype("float32")


# 3. Build the Approximate Index (IVFFlat)
print("Building the Approximate Faiss index...")

nlist = 256  # Number of clusters (cells) to partition the data into.
# A good starting point is round(4 * sqrt(nb))
quantizer = faiss.IndexFlatL2(d)  # The quantizer finds the cluster centroids

# Create the IVF index on the CPU first
cpu_index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)

# Create GPU resources and move the index structure to the GPU
res = faiss.StandardGpuResources()
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

# 4. Train the Index and Add Vectors
print("Training the index on GPU...")
start_time = time.time()
gpu_index.train(xt)  # Training is done on the GPU
end_time = time.time()
print(f"Training completed in {end_time - start_time:.2f} seconds.")

print(f"Adding {nb} vectors to the GPU index...")
start_time = time.time()
gpu_index.add(xb)  # Adding is also done on the GPU
end_time = time.time()
print(f"Vectors added in {end_time - start_time:.2f} seconds.")
print(f"Total vectors in index: {gpu_index.ntotal}")

# 5. Perform the Search with nprobe
k = 5  # Number of nearest neighbors to find

# Set nprobe: the number of nearby clusters to search.
# This is the key parameter for the speed/accuracy trade-off.
# Higher nprobe = more accurate, but slower.
gpu_index.nprobe = 16

print(f"\nSearching with nprobe = {gpu_index.nprobe}...")
start_time = time.time()
distances, indices = gpu_index.search(xq, k)
end_time = time.time()

print(f"GPU search completed in {end_time - start_time:.4f} seconds.")

# 6. Display Results
print("\n--- Search Results ---")
for i in range(5):
    print(f"\nQuery {i}:")
    print(f"  - Nearest neighbor indices: {indices[i]}")
    print(f"  - L2 distances: {distances[i]}")

Generating random data...
Building the Approximate Faiss index...
Training the index on GPU...
Training completed in 0.19 seconds.
Adding 151936 vectors to the GPU index...
Vectors added in 0.10 seconds.
Total vectors in index: 151936

Searching with nprobe = 16...
GPU search completed in 0.0060 seconds.

--- Search Results ---

Query 0:
  - Nearest neighbor indices: [116457 125758 108597 110517  87331]
  - L2 distances: [224.9718  225.40982 227.46252 229.16734 229.42088]

Query 1:
  - Nearest neighbor indices: [130275  84519  98643  85171  93434]
  - L2 distances: [225.35916 225.62064 227.40454 228.76021 229.69127]

Query 2:
  - Nearest neighbor indices: [ 98052 139670  31560  88476 151791]
  - L2 distances: [226.57063 228.79704 229.67294 230.0223  230.08902]

Query 3:
  - Nearest neighbor indices: [ 9385 59675 40047 27327 12967]
  - L2 distances: [230.83804 231.39282 232.74419 232.83736 233.23164]

Query 4:
  - Nearest neighbor indices: [ 71703  95110 141873  76369  33437]
  - L2 dis

### faiss gpu vs matrix mul

In [1]:
import torch
import faiss
import numpy as np
import time

# --- 1. Setup Parameters and Data ---
d = 1536  # Vector dimension
nb = 151936  # Database size
nq = 1000  # Number of query vectors
k = 10  # Number of nearest neighbors to find

# Use a specific GPU
device = "cuda:0"
print(f"Using device: {device}")
print(f"Faiss has access to {faiss.get_num_gpus()} GPUs.")

# Generate random data
np.random.seed(42)
db_vectors_np = np.random.random((nb, d)).astype("float32")
query_vectors_np = np.random.random((nq, d)).astype("float32")

# --- L2 Normalize the data for Cosine Similarity ---
# Normalization is crucial for comparing dot product with cosine similarity
faiss.normalize_L2(db_vectors_np)
faiss.normalize_L2(query_vectors_np)

# Convert numpy arrays to PyTorch tensors and move to GPU
db_vectors_torch = torch.from_numpy(db_vectors_np).to(device)
query_vectors_torch = torch.from_numpy(query_vectors_np).to(device)


# --- 2. Baseline: PyTorch `matmul` (Brute-Force, 100% Accurate) ---
print("\n--- 1. PyTorch `matmul` Baseline ---")
print("Performing brute-force search with PyTorch...")

start_time = time.time()

# Compute dot products (cosine similarity on normalized vectors)
# (nq, d) @ (d, nb) -> (nq, nb)
similarity_matrix = torch.matmul(query_vectors_torch, db_vectors_torch.T)

# Find the top k most similar vectors
# This gives us the scores (distances) and the indices
baseline_distances, baseline_indices = torch.topk(similarity_matrix, k=k, dim=1)

# Ensure all GPU operations are finished before stopping the timer
torch.cuda.synchronize()
end_time = time.time()

print(f"PyTorch search took: {end_time - start_time:.4f} seconds")
# Move results to CPU for later comparison
baseline_indices = baseline_indices.cpu().numpy()


# --- 3. Faiss Exact Search: `IndexFlatIP` ---
print("\n--- 2. Faiss Exact Search (IndexFlatIP) ---")

# Create an index that uses Inner Product (dot product)
index_flat = faiss.IndexFlatIP(d)

# Move the index to the GPU
res = faiss.StandardGpuResources()
gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, index_flat)

# Add the database vectors
gpu_index_flat.add(db_vectors_np)
print(f"Faiss IndexFlatIP created with {gpu_index_flat.ntotal} vectors.")

# Perform the search
start_time = time.time()
D_flat, I_flat = gpu_index_flat.search(query_vectors_np, k)
end_time = time.time()

print(f"Faiss IndexFlatIP search took: {end_time - start_time:.4f} seconds")


# --- 4. Faiss Approximate Search: `IndexIVFFlat` ---
print("\n--- 3. Faiss Approximate Search (IndexIVFFlat) ---")

nlist = 256  # Number of clusters/cells
quantizer = faiss.IndexFlatIP(d)  # The quantizer also uses Inner Product
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

# Move to GPU
gpu_index_ivf = faiss.index_cpu_to_gpu(res, 0, index_ivf)

# Train the index on the database vectors
print("Training IVF index...")
gpu_index_ivf.train(db_vectors_np)

# Add the vectors
gpu_index_ivf.add(db_vectors_np)
print(f"Faiss IndexIVFFlat created with {gpu_index_ivf.ntotal} vectors.")

# --- Search with different `nprobe` values to see the trade-off ---
for nprobe in [1, 4, 16, 64]:
    gpu_index_ivf.nprobe = nprobe

    start_time = time.time()
    D_ivf, I_ivf = gpu_index_ivf.search(query_vectors_np, k)
    end_time = time.time()

    # Calculate recall@k against the ground truth from PyTorch
    # Recall = (number of true neighbors found) / (total true neighbors)
    # We compare the sets of indices to handle order differences
    found_count = 0
    for i in range(nq):
        true_neighbors = set(baseline_indices[i])
        retrieved_neighbors = set(I_ivf[i])
        found_count += len(true_neighbors.intersection(retrieved_neighbors))

    recall = found_count / (nq * k)

    print(f"\n  nprobe = {nprobe}:")
    print(f"    - Search Time: {end_time - start_time:.4f} seconds")
    print(f"    - Recall@{k}: {recall:.4f}")

Using device: cuda:0
Faiss has access to 1 GPUs.

--- 1. PyTorch `matmul` Baseline ---
Performing brute-force search with PyTorch...
PyTorch search took: 0.0672 seconds

--- 2. Faiss Exact Search (IndexFlatIP) ---
Faiss IndexFlatIP created with 151936 vectors.
Faiss IndexFlatIP search took: 0.0128 seconds

--- 3. Faiss Approximate Search (IndexIVFFlat) ---
Training IVF index...
Faiss IndexIVFFlat created with 151936 vectors.

  nprobe = 1:
    - Search Time: 0.0031 seconds
    - Recall@10: 0.0162

  nprobe = 4:
    - Search Time: 0.0184 seconds
    - Recall@10: 0.0632

  nprobe = 16:
    - Search Time: 0.0984 seconds
    - Recall@10: 0.2145

  nprobe = 64:
    - Search Time: 0.4595 seconds
    - Recall@10: 0.6401


### Text generation

In [1]:
import torch
import faiss
import numpy as np
import time
from transformers import AutoModelForCausalLM, AutoTokenizer


# --- ANSI Color Codes for Terminal Highlighting ---
class Colors:
    GREEN = "\033[92m"  # Green for generated text
    ENDC = "\033[0m"  # Reset to default color


# --- Use torch.no_grad() globally for all inference operations ---
with torch.no_grad():
    # --- 1. Setup Model, Tokenizer, and Faiss Indices ---
    print("--- 1. Initializing Environment ---")
    model_name = "Qwen/Qwen2-1.5B-Instruct"

    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token  # Suppress warning

    device = model.device
    print(f"Model loaded on device: {device} with dtype {model.dtype}")

    # Model parameters
    d = model.config.hidden_size
    max_new_tokens = 30

    # Prompts for warmup and benchmarking
    prompts = [
        "What is the capital of France?",
        "Translate 'hello' to Spanish.",
        "The best way to learn is by",
        "Explain the theory of relativity in one sentence.",
        "A list of common fruits:",
        "Once upon a time, in a land far away,",
        "To be or not to be, that is the",
        "The main components of a computer are",
        "Write a short poem about the ocean.",
        "Photosynthesis is the process by which",
    ]

    # --- Pre-build Faiss Indices ---
    print("\n--- Building Faiss Indices (one-time cost) ---")
    db_np = model.lm_head.weight.to(torch.float32).cpu().numpy()
    res = faiss.StandardGpuResources()
    gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(d))
    gpu_index_flat.add(db_np)
    print("Exact Faiss index (IndexFlatIP) is ready.")

    # --- WARMUP PHASE ---
    print("\n--- 2. Starting Warmup Phase ---")
    for p in prompts:
        input_ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        _ = model.generate(input_ids, max_new_tokens=2, do_sample=False)
        outputs = model.model(input_ids, use_cache=False)
        last_token_hidden_state = outputs.last_hidden_state[:, -1, :]
        query_np = last_token_hidden_state.to(torch.float32).cpu().numpy().reshape(1, d)
        _ = gpu_index_flat.search(query_np, k=1)
    torch.cuda.synchronize()
    print("--- Warmup Complete ---\n")

    # Lists to store the full generated texts for comparison
    baseline_full_texts = []
    faiss_full_texts = []

    # --- 3. TIMED BENCHMARK: Standard `model.generate()` ---
    print(
        f"--- 3. TIMED BENCHMARK: Standard `model.generate()` on {len(prompts)} prompts ---"
    )
    torch.cuda.synchronize()
    start_time = time.time()

    for p in prompts:
        input_ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        output_ids = model.generate(
            input_ids, max_new_tokens=max_new_tokens, do_sample=False
        )
        decoded_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        baseline_full_texts.append(decoded_text)

    torch.cuda.synchronize()
    end_time = time.time()
    total_baseline_time = end_time - start_time
    print("--- Baseline Benchmark Complete ---")

    # --- 4. TIMED BENCHMARK: Manual Faiss Method ---
    print(
        f"\n--- 4. TIMED BENCHMARK: Manual Faiss Method on {len(prompts)} prompts ---"
    )
    torch.cuda.synchronize()
    start_time = time.time()

    for p in prompts:
        input_ids_manual = tokenizer(p, return_tensors="pt").input_ids.to(device)
        past_key_values = None
        generated_ids = []

        for i in range(max_new_tokens):
            current_input = (
                input_ids_manual
                if i == 0
                else torch.tensor([[generated_ids[-1]]], device=device)
            )
            outputs = model.model(
                current_input, past_key_values=past_key_values, use_cache=True
            )
            last_token_hidden_state = outputs.last_hidden_state[:, -1, :]
            past_key_values = outputs.past_key_values
            query_np = (
                last_token_hidden_state.to(torch.float32).cpu().numpy().reshape(1, d)
            )
            _, I = gpu_index_flat.search(query_np, k=1)
            next_token_id = I[0][0]
            generated_ids.append(next_token_id)

        full_sequence_ids = input_ids_manual.tolist()[0] + generated_ids
        decoded_text = tokenizer.decode(full_sequence_ids, skip_special_tokens=True)
        faiss_full_texts.append(decoded_text)

    torch.cuda.synchronize()
    end_time = time.time()
    total_faiss_time = end_time - start_time
    print("--- Manual Faiss Benchmark Complete ---")

    # --- 5. Final Performance Results ---
    print("\n\n" + "=" * 50)
    print(" " * 10 + "FINAL PERFORMANCE RESULTS")
    print("=" * 50)
    print(
        f"Benchmark run on {len(prompts)} prompts, generating {max_new_tokens} new tokens each."
    )
    print("-" * 50)
    print(f"Baseline `model.generate()`:")
    print(f"  Total Time: {total_baseline_time:.4f} seconds")
    print(f"  Avg. Time per Prompt: {total_baseline_time / len(prompts):.4f} seconds")
    print("-" * 50)
    print(f"Manual Faiss Method:")
    print(f"  Total Time: {total_faiss_time:.4f} seconds")
    print(f"  Avg. Time per Prompt: {total_faiss_time / len(prompts):.4f} seconds")
    print("-" * 50)

    speed_diff = (total_faiss_time - total_baseline_time) / total_baseline_time * 100
    if speed_diff > 0:
        print(
            f"\nConclusion: The manual Faiss method was {speed_diff:.2f}% SLOWER than the baseline."
        )
    else:
        print(
            f"\nConclusion: The manual Faiss method was {-speed_diff:.2f}% FASTER than the baseline."
        )
    print("=" * 50)

    # --- 6. Qualitative Text Comparison ---
    print("\n\n" + "=" * 50)
    print(" " * 8 + "QUALITATIVE TEXT COMPARISON")
    print("=" * 50)
    divergence_count = 0
    for i in range(len(prompts)):
        prompt_text = prompts[i]
        baseline_full = baseline_full_texts[i]
        faiss_full = faiss_full_texts[i]

        # Isolate the generated part of the text for highlighting
        baseline_gen = (
            baseline_full[len(prompt_text) :]
            if baseline_full.startswith(prompt_text)
            else " [Output Error]"
        )
        faiss_gen = (
            faiss_full[len(prompt_text) :]
            if faiss_full.startswith(prompt_text)
            else " [Output Error]"
        )

        print(f"\n----- Prompt {i+1} -----")
        print(
            f"  Baseline `generate()`: {prompt_text}{Colors.GREEN}{baseline_gen}{Colors.ENDC}"
        )
        print(
            f"  Manual Faiss Method  : {prompt_text}{Colors.GREEN}{faiss_gen}{Colors.ENDC}"
        )

        if baseline_full != faiss_full:
            divergence_count += 1
            print("  *** NOTE: Outputs have diverged! ***")

    print("\n" + "-" * 50)
    print("--- Comparison Summary ---")
    print(
        f"Number of prompts where generated text diverged: {divergence_count} out of {len(prompts)}"
    )
    print("=" * 50)

--- 1. Initializing Environment ---
Model loaded on device: cuda:0 with dtype torch.float16

--- Building Faiss Indices (one-time cost) ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Exact Faiss index (IndexFlatIP) is ready.

--- 2. Starting Warmup Phase ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

--- Warmup Complete ---

--- 3. TIMED BENCHMARK: Standard `model.generate()` on 10 prompts ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

--- Baseline Benchmark Complete ---

--- 4. TIMED BENCHMARK: Manual Faiss Method on 10 prompts ---
--- Manual Faiss Benchmark Complete ---


          FINAL PERFORMANCE RESULTS
Benchmark run on 10 prompts, generating 30 new tokens each.
--------------------------------------------------
Baseline `model.generate()`:
  Total Time: 2.4462 seconds
  Avg. Time per Prompt: 0.2446 seconds
--------------------------------------------------
Manual Faiss Method:
  Total Time: 2.5498 seconds
  Avg. Time per Prompt: 0.2550 seconds
--------------------------------------------------

Conclusion: The manual Faiss method was 4.24% SLOWER than the baseline.


        QUALITATIVE TEXT COMPARISON

----- Prompt 1 -----
  Baseline `generate()`: What is the capital of France? Paris. 

The answer is: Paris. 

Justification: Paris is the capital city of France, as stated in the question and confirmed by common
  Manual Faiss Method  : What is the capital of France? Paris. 

The capital of France is Paris. 

P

### Optimize generation

In [1]:
import torch
import faiss
import numpy as np
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

# Attempt to import RAPIDS cuml. If it fails, the corresponding benchmark will be skipped.
try:
    from cuml.neighbors import NearestNeighbors as cuMLNearestNeighbors

    cuml_available = True
except ImportError:
    print("Warning: RAPIDS cuml could not be imported. Skipping the cuml benchmark.")
    print(
        "         To run this benchmark, please install RAPIDS: https://rapids.ai/start.html"
    )
    cuml_available = False


# --- ANSI Color Codes for Terminal Highlighting ---
class Colors:
    GREEN = "\033[92m"  # Baseline
    MAGENTA = "\033[95m"  # Faiss
    YELLOW = "\033[93m"  # cuml
    ENDC = "\033[0m"  # Reset to default color


# --- Use torch.no_grad() globally for all inference operations ---
with torch.no_grad():
    # --- 1. Setup Environment, Model, and Tokenizer ---
    print("--- 1. Initializing Environment ---")
    model_name = "Qwen/Qwen2-1.5B-Instruct"

    try:
        # Load the model in bfloat16 for best performance on compatible hardware
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16, device_map="auto"
        )
    except Exception as e:
        print(f"Could not load model in bfloat16 ({e}). Falling back to float16.")
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    device = model.device
    print(f"Model loaded on device: {device} with dtype {model.dtype}")

    # Model parameters
    d = model.config.hidden_size
    max_new_tokens = 30
    prompts = [
        "What is the capital of France?",
        "Translate 'hello' to Spanish.",
        "The best way to learn is by",
        "Explain the theory of relativity in one sentence.",
        "A list of common fruits:",
        "Once upon a time, in a land far away,",
        "To be or not to be, that is the",
        "The main components of a computer are",
        "Write a short poem about the ocean.",
        "Photosynthesis is the process by which",
    ]

    # --- 2. Pre-build Approximate Nearest Neighbor (ANN) Indices ---
    print("\n--- 2. Building ANN Indices (one-time cost) ---")
    # The model's lm_head is now bfloat16. We must convert it to a supported dtype
    # (float32 for best precision) to build the ANN indices.
    db_vectors_gpu_f32 = model.lm_head.weight.to(torch.float32)

    # --- Index 1: Faiss IndexIVFFlat (GPU) ---
    res = faiss.StandardGpuResources()
    n_clusters = 1024
    nprobe = 8
    quantizer = faiss.IndexFlatIP(d)
    gpu_index_ivf = faiss.IndexIVFFlat(
        quantizer, d, n_clusters, faiss.METRIC_INNER_PRODUCT
    )
    gpu_index_ivf = faiss.index_cpu_to_gpu(res, 0, gpu_index_ivf)
    print("Training Faiss Hierarchical Index...")
    # The Faiss Python API requires a CPU numpy array for setup.
    gpu_index_ivf.train(db_vectors_gpu_f32.cpu().numpy())
    gpu_index_ivf.add(db_vectors_gpu_f32.cpu().numpy())
    gpu_index_ivf.nprobe = nprobe
    print(f"Faiss index (GPU) is ready. nprobe={nprobe}")

    # --- Index 2: RAPIDS cuml NearestNeighbors (GPU) ---
    if cuml_available:
        print("Building cuml IVF-Flat Index...")
        cuml_index = cuMLNearestNeighbors(
            n_neighbors=1,
            metric="inner_product",
            algo="ivf-flat",
            ivf_flat_n_lists=n_clusters,
            ivf_flat_n_probes=nprobe,
        )
        # cuml's key advantage: .fit() can take a GPU tensor directly!
        cuml_index.fit(db_vectors_gpu_f32)
        print(f"cuml index (GPU) is ready. n_probes={nprobe}")

    # --- 3. Warmup Phase ---
    print("\n--- 3. Starting Warmup Phase ---")
    # ... (A proper warmup would call all benchmarked functions once) ...
    print("--- Warmup Complete ---\n")

    # Lists to store the full generated texts for comparison
    baseline_full_texts = []
    faiss_ann_full_texts = []
    cuml_ann_full_texts = []

    # --- 4. BENCHMARK: Standard `model.generate()` (Baseline) ---
    print(
        f"--- 4. BENCHMARK: Standard `model.generate()` (Baseline, model dtype={model.dtype}) ---"
    )
    torch.cuda.synchronize()
    start_time = time.time()
    for p in prompts:
        input_ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        output_ids = model.generate(
            input_ids, max_new_tokens=max_new_tokens, do_sample=False
        )
        baseline_full_texts.append(
            tokenizer.decode(output_ids[0], skip_special_tokens=True)
        )
    torch.cuda.synchronize()
    total_baseline_time = time.time() - start_time
    print("--- Baseline Benchmark Complete ---")

    # --- 5. BENCHMARK: Faiss Hierarchical Method (GPU ANN) ---
    print(f"\n--- 5. BENCHMARK: Faiss Method (bfloat16 model -> CPU f32 query) ---")
    torch.cuda.synchronize()
    start_time = time.time()
    for p in prompts:
        input_ids_manual = tokenizer(p, return_tensors="pt").input_ids.to(device)
        past_key_values = None
        generated_ids_list = []
        current_input = input_ids_manual
        for _ in range(max_new_tokens):
            outputs = model.model(
                current_input, past_key_values=past_key_values, use_cache=True
            )
            last_token_hidden_state = outputs.last_hidden_state[
                :, -1, :
            ]  # This is bfloat16
            past_key_values = outputs.past_key_values

            # Convert bfloat16 query to float32 and move to CPU for Faiss's Python API
            query_np = last_token_hidden_state.to(torch.float32).cpu().numpy()

            _, I = gpu_index_ivf.search(query_np, k=1)
            next_token_id = I[0][0]
            generated_ids_list.append(next_token_id)
            current_input = torch.tensor(
                [[next_token_id]], device=device, dtype=torch.long
            )
        full_ids = torch.cat(
            [input_ids_manual[0], torch.tensor(generated_ids_list, device=device)]
        )
        faiss_ann_full_texts.append(
            tokenizer.decode(full_ids, skip_special_tokens=True)
        )
    torch.cuda.synchronize()
    total_faiss_time = time.time() - start_time
    print("--- Faiss Hierarchical Benchmark Complete ---")

    # --- 6. BENCHMARK: cuml Method (GPU ANN) ---
    if cuml_available:
        print(f"\n--- 6. BENCHMARK: cuml Method (bfloat16 model -> GPU f32 query) ---")
        torch.cuda.synchronize()
        start_time = time.time()
        for p in prompts:
            input_ids_manual = tokenizer(p, return_tensors="pt").input_ids.to(device)
            past_key_values = None
            generated_ids_list = []
            current_input = input_ids_manual
            for _ in range(max_new_tokens):
                outputs = model.model(
                    current_input, past_key_values=past_key_values, use_cache=True
                )
                last_token_hidden_state = outputs.last_hidden_state[
                    :, -1, :
                ]  # This is bfloat16
                past_key_values = outputs.past_key_values

                # Perform efficient ON-DEVICE conversion from bfloat16 to float32 for cuml
                query_gpu_f32 = last_token_hidden_state.to(torch.float32)

                # Search the cuml index directly with the GPU tensor
                _, indices = cuml_index.kneighbors(query_gpu_f32)
                next_token_id = indices[0][0].item()  # .item() gets the scalar value

                generated_ids_list.append(next_token_id)
                current_input = torch.tensor(
                    [[next_token_id]], device=device, dtype=torch.long
                )
            full_ids = torch.cat(
                [input_ids_manual[0], torch.tensor(generated_ids_list, device=device)]
            )
            cuml_ann_full_texts.append(
                tokenizer.decode(full_ids, skip_special_tokens=True)
            )
        torch.cuda.synchronize()
        total_cuml_time = time.time() - start_time
        print("--- cuml Benchmark Complete ---")

    # --- 7. Final Performance Results ---
    print("\n\n" + "=" * 65)
    print(f" " * 15 + f"FINAL PERFORMANCE RESULTS (Model: {model.dtype})")
    print("=" * 65)
    print(
        f"Benchmark run on {len(prompts)} prompts, generating {max_new_tokens} new tokens each."
    )
    print("-" * 65)
    print(f"Baseline `model.generate()`:")
    print(
        f"  Total Time: {total_baseline_time:.4f}s | Avg: {total_baseline_time / len(prompts):.4f}s"
    )
    print(f"Faiss ANN (GPU Search, CPU Query Transfer):")
    print(
        f"  Total Time: {total_faiss_time:.4f}s | Avg: {total_faiss_time / len(prompts):.4f}s"
    )
    if cuml_available:
        print(f"cuml ANN (GPU Search, GPU Query Transfer):")
        print(
            f"  Total Time: {total_cuml_time:.4f}s | Avg: {total_cuml_time / len(prompts):.4f}s"
        )
    print("=" * 65)

    # --- 8. Qualitative Text Comparison ---
    print("\n\n" + "=" * 65)
    print(" " * 18 + "QUALITATIVE TEXT COMPARISON")
    print("=" * 65)
    for i in range(len(prompts)):
        prompt_text = prompts[i]

        # Isolate the generated part of the text for cleaner comparison
        baseline_gen = baseline_full_texts[i][len(prompt_text) :]
        faiss_gen = faiss_ann_full_texts[i][len(prompt_text) :]

        print(f"\n----- Prompt {i+1}: '{prompt_text}' -----")
        print(f"  Baseline (Exact): ...{Colors.GREEN}{baseline_gen}{Colors.ENDC}")
        print(f"  Faiss (Approx):   ...{Colors.MAGENTA}{faiss_gen}{Colors.ENDC}")
        if cuml_available:
            cuml_gen = cuml_ann_full_texts[i][len(prompt_text) :]
            print(f"  cuml (Approx):    ...{Colors.YELLOW}{cuml_gen}{Colors.ENDC}")

        if baseline_full_texts[i] != faiss_ann_full_texts[i]:
            print("  -> Baseline and Faiss diverged (expected).")
        if cuml_available and faiss_ann_full_texts[i] != cuml_ann_full_texts[i]:
            print(
                "  -> Faiss and cuml diverged (possible due to implementation differences)."
            )

    print("\n" + "=" * 65)

--- 1. Initializing Environment ---
Model loaded on device: cuda:0 with dtype torch.bfloat16

--- 2. Building ANN Indices (one-time cost) ---
Training Faiss Hierarchical Index...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Faiss index (GPU) is ready. nprobe=8
Building cuml IVF-Flat Index...
cuml index (GPU) is ready. n_probes=8

--- 3. Starting Warmup Phase ---
--- Warmup Complete ---

--- 4. BENCHMARK: Standard `model.generate()` (Baseline, model dtype=torch.bfloat16) ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

--- Baseline Benchmark Complete ---

--- 5. BENCHMARK: Faiss Method (bfloat16 model -> CPU f32 query) ---
--- Faiss Hierarchical Benchmark Complete ---

--- 6. BENCHMARK: cuml Method (bfloat16 model -> GPU f32 query) ---
--- cuml Benchmark Complete ---


               FINAL PERFORMANCE RESULTS (Model: torch.bfloat16)
Benchmark run on 10 prompts, generating 30 new tokens each.
-----------------------------------------------------------------
Baseline `model.generate()`:
  Total Time: 2.5235s | Avg: 0.2524s
Faiss ANN (GPU Search, CPU Query Transfer):
  Total Time: 2.2222s | Avg: 0.2222s
cuml ANN (GPU Search, GPU Query Transfer):
  Total Time: 2.7028s | Avg: 0.2703s


                  QUALITATIVE TEXT COMPARISON

----- Prompt 1: 'What is the capital of France?' -----
  Baseline (Exact): ... Paris. 

The answer is: Paris.
  Faiss (Approx):   ... Paris. 

The correct spelling of the French city name is "Pariis". 

The French language is a West Germanic language that is spoken
  cuml (Appr

### попытка сохранить качество

In [1]:
import torch
import faiss
import numpy as np
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

# Import cuml and check for availability
try:
    from cuml.neighbors import NearestNeighbors as cuMLNearestNeighbors
    import cupy

    cuml_available = True
except ImportError:
    print("Warning: RAPIDS cuml or cupy not found. Skipping cuml benchmark.")
    cuml_available = False


# --- ANSI Color Codes ---
class Colors:
    GREEN = "\033[92m"  # Baseline
    BLUE = "\033[94m"  # Hybrid Search (High Accuracy)
    ENDC = "\033[0m"


# --- Use torch.no_grad() globally ---
with torch.no_grad():
    # --- 1. Setup ---
    print("--- 1. Initializing Environment ---")
    model_name = "Qwen/Qwen2-1.5B-Instruct"
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16, device_map="auto"
        )
    except Exception as e:
        print(f"Could not load in bfloat16 ({e}), falling back to float16.")
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    device = model.device
    print(f"Model loaded on device: {device} with dtype {model.dtype}")

    # Parameters
    d = model.config.hidden_size
    max_new_tokens = 30
    prompts = [
        "What is the capital of France?",
        "Translate 'hello' to Spanish.",
        "The best way to learn is by",
        "Explain the theory of relativity in one sentence.",
        "A list of common fruits:",
        "Once upon a time, in a land far away,",
        "To be or not to be, that is the",
        "The main components of a computer are",
        "Write a short poem about the ocean.",
        "Photosynthesis is the process by which",
    ]

    # --- 2. Pre-build Indices ---
    print("\n--- 2. Building ANN Indices ---")
    db_vectors_gpu_f32 = model.lm_head.weight.to(torch.float32)

    if cuml_available:
        # Parameters for Candidate Retrieval
        k_candidates = 64
        n_clusters = 1024
        nprobe = 128

        print("Building cuml IVF-Flat Index for Hybrid Search...")
        cuml_index_hybrid = cuMLNearestNeighbors(
            n_neighbors=k_candidates,
            metric="inner_product",
            algo="ivf-flat",
            ivf_flat_n_lists=n_clusters,
            ivf_flat_n_probes=nprobe,
        )
        cuml_index_hybrid.fit(db_vectors_gpu_f32)
        print(f"cuml index ready. k={k_candidates}, n_probes={nprobe}")

    # --- 3. Warmup ---
    print("\n--- 3. Warmup Complete ---\n")

    baseline_full_texts = []
    hybrid_search_full_texts = []

    # --- 4. BENCHMARK: Standard `model.generate()` (Gold Standard for Quality) ---
    print(f"--- 4. BENCHMARK: Standard `model.generate()` (Baseline) ---")
    torch.cuda.synchronize()
    start_time = time.time()
    for p in prompts:
        input_ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        output_ids = model.generate(
            input_ids, max_new_tokens=max_new_tokens, do_sample=False
        )
        baseline_full_texts.append(
            tokenizer.decode(output_ids[0], skip_special_tokens=True)
        )
    torch.cuda.synchronize()
    total_baseline_time = time.time() - start_time
    print("--- Baseline Benchmark Complete ---")

    # --- 5. BENCHMARK: Hybrid Search (High Accuracy) ---
    if cuml_available:
        print(f"\n--- 5. BENCHMARK: Hybrid Search (ANN Candidates + Exact Rerank) ---")
        torch.cuda.synchronize()
        start_time = time.time()

        embedding_matrix = model.lm_head.weight

        for p in prompts:
            input_ids_manual = tokenizer(p, return_tensors="pt").input_ids.to(device)
            past_key_values = None
            generated_ids_list = []
            current_input = input_ids_manual

            for _ in range(max_new_tokens):
                outputs = model.model(
                    current_input, past_key_values=past_key_values, use_cache=True
                )
                hidden_state = outputs.last_hidden_state[:, -1, :]
                past_key_values = outputs.past_key_values

                # --- Step 1: ANN Candidate Retrieval ---
                query_gpu_f32 = hidden_state.to(torch.float32)
                _, candidate_indices_cupy = cuml_index_hybrid.kneighbors(query_gpu_f32)

                # *** THE FIX IS HERE ***
                # Convert the CuPy array of indices to a PyTorch tensor for indexing.
                # This is a zero-copy operation on the GPU.
                candidate_indices = torch.as_tensor(
                    candidate_indices_cupy, device=device
                ).flatten()

                # --- Step 2: Exact Re-ranking on GPU ---
                candidate_vectors = embedding_matrix[candidate_indices]
                rerank_scores = torch.matmul(hidden_state, candidate_vectors.t())

                # --- Step 3: Final ArgMax ---
                best_candidate_local_idx = torch.argmax(rerank_scores)
                next_token_id = candidate_indices[best_candidate_local_idx].item()

                generated_ids_list.append(next_token_id)
                current_input = torch.tensor(
                    [[next_token_id]], device=device, dtype=torch.long
                )

            full_ids = torch.cat(
                [input_ids_manual[0], torch.tensor(generated_ids_list, device=device)]
            )
            hybrid_search_full_texts.append(
                tokenizer.decode(full_ids, skip_special_tokens=True)
            )

        torch.cuda.synchronize()
        total_hybrid_time = time.time() - start_time
        print("--- Hybrid Search Benchmark Complete ---")

    # --- 6. Final Results & Qualitative Comparison ---
    print("\n\n" + "=" * 65)
    print(f" " * 15 + f"FINAL PERFORMANCE & ACCURACY RESULTS")
    print("=" * 65)
    print(f"Baseline `model.generate()`:")
    print(
        f"  Total Time: {total_baseline_time:.4f}s | Avg: {total_baseline_time / len(prompts):.4f}s"
    )
    if cuml_available:
        print(f"Hybrid Search (cuml k={k_candidates} + Rerank):")
        print(
            f"  Total Time: {total_hybrid_time:.4f}s | Avg: {total_hybrid_time / len(prompts):.4f}s"
        )
    print("=" * 65)

    print("\n\n" + "=" * 65)
    print(" " * 18 + "QUALITATIVE TEXT COMPARISON")
    print("=" * 65)
    divergence_count = 0
    if cuml_available:
        for i in range(len(prompts)):
            prompt_text = prompts[i]
            baseline_gen = baseline_full_texts[i][len(prompt_text) :]
            hybrid_gen = hybrid_search_full_texts[i][len(prompt_text) :]

            print(f"\n----- Prompt {i+1}: '{prompt_text}' -----")
            print(f"  Baseline (Exact): ...{Colors.GREEN}{baseline_gen}{Colors.ENDC}")
            print(f"  Hybrid (Accurate):...{Colors.BLUE}{hybrid_gen}{Colors.ENDC}")

            if baseline_full_texts[i] != hybrid_search_full_texts[i]:
                divergence_count += 1
                print("  -> *** NOTE: Outputs have diverged! ***")

    print("\n" + "-" * 65)
    print("--- Accuracy Summary ---")
    if cuml_available:
        print(
            f"Number of prompts where generated text diverged: {divergence_count} out of {len(prompts)}"
        )
        if divergence_count == 0:
            print(
                "Conclusion: The Hybrid Search achieved IDENTICAL output to the baseline!"
            )
        else:
            print(
                f"Conclusion: Accuracy improved, but did not perfectly match baseline."
            )
            print(f"To improve further, try increasing k_candidates or nprobe.")
    print("=" * 65)

--- 1. Initializing Environment ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model loaded on device: cuda:0 with dtype torch.bfloat16

--- 2. Building ANN Indices ---
Building cuml IVF-Flat Index for Hybrid Search...
cuml index ready. k=64, n_probes=128

--- 3. Warmup Complete ---

--- 4. BENCHMARK: Standard `model.generate()` (Baseline) ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

--- Baseline Benchmark Complete ---

--- 5. BENCHMARK: Hybrid Search (ANN Candidates + Exact Rerank) ---
--- Hybrid Search Benchmark Complete ---


               FINAL PERFORMANCE & ACCURACY RESULTS
Baseline `model.generate()`:
  Total Time: 2.5465s | Avg: 0.2546s
Hybrid Search (cuml k=64 + Rerank):
  Total Time: 2.7355s | Avg: 0.2736s


                  QUALITATIVE TEXT COMPARISON

----- Prompt 1: 'What is the capital of France?' -----
  Baseline (Exact): ... Paris. 

The answer is: Paris.
  Hybrid (Accurate):... Paris. 

The capital of France is Paris. 

Paris is the capital of France. 

Paris is the capital of France. 

Paris is the
  -> *** NOTE: Outputs have diverged! ***

----- Prompt 2: 'Translate 'hello' to Spanish.' -----
  Baseline (Exact): ... Hello in Spanish is "hola". 

In this case, the word "hello" refers to a greeting or informal way of saying "hi",
  Hybrid (Accurate):... Hello in Spanish is "hola". 

In Spanish, "hello" is a common greeting and is used to say "hell

### Может изначальные вектора в модели слишком близко между собой?

In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Use torch.no_grad() globally ---
with torch.no_grad():
    # --- 1. Setup Model and Tokenizer ---
    print("--- 1. Initializing Environment ---")
    model_name = "Qwen/Qwen2-1.5B-Instruct"
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16, device_map="auto"
        )
    except Exception as e:
        print(f"Could not load in bfloat16 ({e}), falling back to float16.")
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    device = model.device
    print(f"Model loaded on device: {device} with dtype {model.dtype}")

    # --- 2. The Experiment ---
    prompt = "What is the capital of France?"
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    print(f"\n--- 2. Analyzing the FIRST generated token for prompt: '{prompt}' ---")

    # Get the hidden state for the last token of the prompt
    outputs = model.model(input_ids)
    hidden_state = outputs.last_hidden_state[:, -1, :]  # Shape: [1, hidden_size]
    embedding_matrix = model.lm_head.weight  # Shape: [vocab_size, hidden_size]

    # --- 3. Calculate Ground Truth Logits ---
    print("\n--- 3. Calculating Ground Truth Logits (Exact MatMul) ---")
    exact_logits = torch.matmul(
        hidden_state, embedding_matrix.t()
    )  # Shape: [1, vocab_size]

    # --- 4. Analyze the Top Contenders and Their Scores ---
    k = 10
    top_k_logits, top_k_indices = torch.topk(exact_logits, k)

    # Flatten for easier processing
    top_k_logits = top_k_logits.flatten()
    top_k_indices = top_k_indices.flatten()

    print(f"\n--- 4. Top {k} Token Candidates and Their Logit Scores ---")
    print("A small difference between Top-1 and Top-2 scores confirms the hypothesis.")
    print("-" * 60)
    for i in range(k):
        token_id = top_k_indices[i].item()
        logit_score = top_k_logits[i].item()
        decoded_token = tokenizer.decode(token_id)
        print(
            f"Rank {i+1}: Token='{decoded_token}' (ID: {token_id}), Logit Score: {logit_score:.4f}"
        )
    print("-" * 60)

    # Let's see this as probabilities to make it more intuitive
    top_k_probs = F.softmax(top_k_logits, dim=-1)
    print("\nSame data viewed as probabilities (after softmax on just these top K):")
    for i in range(k):
        print(
            f"Rank {i+1}: Token='{tokenizer.decode(top_k_indices[i].item())}', Probability: {top_k_probs[i].item():.4f}"
        )
    print("-" * 60)

    # --- 5. Directly Compare the Top 2 Vectors ---
    print("\n--- 5. Direct Vector Comparison (Cosine Similarity) ---")
    print("A similarity score very close to 1.0 is definitive proof.")

    top_1_token_id = top_k_indices[0]
    top_2_token_id = top_k_indices[1]

    vector1 = embedding_matrix[top_1_token_id]
    vector2 = embedding_matrix[top_2_token_id]

    # Calculate cosine similarity
    similarity = F.cosine_similarity(vector1.unsqueeze(0), vector2.unsqueeze(0))

    print(f"Vector for Top-1 token '{tokenizer.decode(top_1_token_id)}'")
    print(f"Vector for Top-2 token '{tokenizer.decode(top_2_token_id)}'")
    print(f"\nCosine Similarity between them: {similarity.item():.6f}")
    print("=" * 60)

--- 1. Initializing Environment ---
Model loaded on device: cuda:0 with dtype torch.bfloat16

--- 2. Analyzing the FIRST generated token for prompt: 'What is the capital of France?' ---

--- 3. Calculating Ground Truth Logits (Exact MatMul) ---

--- 4. Top 10 Token Candidates and Their Logit Scores ---
A small difference between Top-1 and Top-2 scores confirms the hypothesis.
------------------------------------------------------------
Rank 1: Token=' Paris' (ID: 12095), Logit Score: 16.0000
Rank 2: Token=' The' (ID: 576), Logit Score: 15.6250
Rank 3: Token=' 

' (ID: 4710), Logit Score: 15.1875
Rank 4: Token=' 
' (ID: 715), Logit Score: 14.6250
Rank 5: Token=' It' (ID: 1084), Logit Score: 14.5000
Rank 6: Token=' A' (ID: 362), Logit Score: 13.6250
Rank 7: Token=' What' (ID: 3555), Logit Score: 13.4375
Rank 8: Token=' I' (ID: 358), Logit Score: 12.5625
Rank 9: Token=' ' (ID: 220), Logit Score: 12.4375
Rank 10: Token=' Is' (ID: 2160), Logit Score: 12.3125
--------------------------------

### Можем ли мы построить лосс для моделирования, в котором будет заложено свойство намеренной удаленности друг от друга

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler
from datasets import load_dataset
from tqdm.auto import tqdm
import random

# --- 1. Configuration ---
model_name = "Qwen/Qwen2-1.5B-Instruct"
# IMPORTANT: This will create a new, fine-tuned model directory
new_model_name = "Qwen2-1.5B-Instruct-MetricTuned"
dataset_name = "wikitext"
dataset_config = "wikitext-2-raw-v1"

# Training Hyperparameters
num_epochs = 1
batch_size = 2  # Keep low to fit on consumer GPUs
max_seq_length = 256
learning_rate = 5e-6

# Metric Learning Hyperparameters
triplet_loss_margin = 0.5  # The "buffer zone" size
triplet_loss_alpha = 0.1  # How much weight to give the margin loss

# --- 2. Load Model and Tokenizer ---
# We need the model for training, so no torch.no_grad() here
print("--- Loading Model for Fine-Tuning ---")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Move model to GPU. accelerate will handle this better in a real script.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model on device: {device}")

# --- 3. Prepare Dataset ---
print("--- Preparing Dataset ---")
raw_datasets = load_dataset(dataset_name, dataset_config)


def tokenize_function(examples):
    # Tokenize the texts
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
        return_tensors="pt",
    )


tokenized_datasets = raw_datasets.map(
    tokenize_function, batched=True, remove_columns=["text"]
)
# We only need the training set for this example
tokenized_datasets.set_format("torch")
train_dataset = (
    tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
)  # Use a small subset
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


# --- 4. Setup Optimizer and Scheduler ---
optimizer = AdamW(model.parameters(), lr=learning_rate)
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

# --- 5. The Fine-Tuning Loop with Compound Loss ---
model.train()
progress_bar = tqdm(range(num_training_steps))
print("\n--- Starting Fine-Tuning with Compound Loss ---")

for epoch in range(num_epochs):
    for batch in train_dataloader:
        # Move batch to the correct device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # The labels are the same as the input_ids for language modeling
        labels = input_ids.clone()

        # Get model outputs
        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True,  # We need the hidden states
        )

        # --- a) Standard Cross-Entropy Loss ---
        # The model conveniently calculates this for us if we provide labels
        loss_ce = outputs.loss

        # --- b) Triplet Margin Loss ---
        logits = outputs.logits
        hidden_states = outputs.hidden_states[-1]  # Last layer hidden states
        embedding_matrix = model.get_output_embeddings().weight

        # We calculate triplet loss for each position in the sequence
        total_triplet_loss = 0
        num_tokens = 0

        # Iterate over each item in the batch
        for i in range(hidden_states.size(0)):
            # Iterate over each token in the sequence (except the last one)
            for j in range(hidden_states.size(1) - 1):
                if attention_mask[i, j] == 0:
                    continue  # Skip padding

                # Anchor: The hidden state at this position
                anchor_h = hidden_states[i, j, :]

                # Positive: The embedding of the correct next token
                positive_id = labels[i, j + 1]
                # Ignore padding tokens as positives
                if positive_id == tokenizer.pad_token_id:
                    continue

                positive_vec = embedding_matrix[positive_id]
                score_p = torch.dot(anchor_h, positive_vec)

                # Negative: Find a "hard negative" - a wrong token with a high score
                # We exclude the correct token from the search for a negative
                logits[i, j, positive_id] = -float("Inf")
                hard_negative_id = torch.argmax(logits[i, j])
                negative_vec = embedding_matrix[hard_negative_id]
                score_n = torch.dot(anchor_h, negative_vec)

                # Calculate the triplet loss for this token
                triplet_loss = F.relu(triplet_loss_margin - (score_p - score_n))
                total_triplet_loss += triplet_loss
                num_tokens += 1

        # Average the triplet loss over all tokens in the batch
        loss_triplet = total_triplet_loss / (num_tokens + 1e-9)

        # --- c) Combine the Losses ---
        total_loss = loss_ce + triplet_loss_alpha * loss_triplet

        # --- Standard Training Steps ---
        total_loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        progress_bar.update(1)
        progress_bar.set_description(
            f"Total Loss: {total_loss.item():.4f}, CE: {loss_ce.item():.4f}, Triplet: {loss_triplet.item():.4f}"
        )

# --- 6. Save the Fine-Tuned Model ---
print("\n--- Saving the fine-tuned model ---")
model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)
print(f"Model saved to '{new_model_name}'")